##### Testing Sampling Against Papers
This will be used to evaluate the performance of the package against reference peer reviewed papers.

!["alt"](../images/franks2014Table3.png "Franks2014 p848 Table3")

In [3]:
import networkx as nx
from Influence_Modelling import XsvNetwork, NetworkMetrics, Sampling

import matplotlib.pyplot as plt
import pandas as pd

In [4]:
enron = XsvNetwork.XsvNetwork("", "", "enron")
enron_nx = enron.get_nx()

airports = XsvNetwork.XsvNetwork("", "", "airports")
airports_nx = airports.get_nx()

twitter = XsvNetwork.XsvNetwork("", "", "twitter")
twitter_nx = twitter.get_nx()

p2p = XsvNetwork.XsvNetwork("", "", "p2p")
p2p_nx = p2p.get_nx()

enron_nx = enron.get_nx()
airports_nx = airports.get_nx()
twitter_nx = twitter.get_nx()
p2p_nx = p2p.get_nx()

The below function will be used to generate the data that is held in the rows of the above table.

In [5]:
def generate_row(input_graph: nx.DiGraph):
  return [
    NetworkMetrics.s_1_number_nodes(input_graph), 
    NetworkMetrics.s_2_number_edges(input_graph),
    NetworkMetrics.s_3_mean_degree(input_graph),
    NetworkMetrics.s_4_clustering_coefficient(input_graph),
    NetworkMetrics.s_6_centralization(input_graph)  
  ]

def repeated_trial(func, repeats): 
  pass

def generate_rows_statistics(input_graph, repeats, significant_figures = 4):
  pass

In [6]:
table = []
for name, data in [("enron", enron_nx), ("p2p",p2p_nx), ("airport", airports_nx)]:
  table.append([name]+generate_row(data))
print(table)
raw_data = pd.DataFrame(table, columns=["Dataset", "Nodes", "Edges", "Mean Degree", "Clustering Coefficient", "Centralization"])
raw_data

[['enron', 36692, 183831, 10.020222391802028, 0.49698255959950505, 0.007250116701537263], ['p2p', 10876, 39994, 7.354542111070247, 0.006217532771466019, 0.0018991065735949049], ['airport', 1574, 17215, 21.874205844980942, 0.504152539209577, 0.004333863926850598]]


,Dataset,Nodes,Edges,Mean Degree,Clustering Coefficient,Centralization
0,enron,36692,183831,10.020222,0.496983,0.007250
1,p2p,10876,39994,7.354542,0.006218,0.001899
2,airport,1574,17215,21.874206,0.504153,0.004334


Compare the sampling methods on the enron dataset as an exmaple and generalise to allow for any sample

In [7]:
import random
from typing import Any, Set

def degree_weighted_random_choice():
    pass

def mhrw(base_graph: nx.DiGraph, n) -> nx.DiGraph:
    # Practical Recommendations on Crawling social networks (page 5)
    subgraph_nodes = set()
    v = random_sample([v for v in base_graph.nodes()], 1)[0]
    while len([v for v in subgraph_nodes]) < n:
        neighbours = [node for node in base_graph.adj[v]]
        random.shuffle(neighbours)
        w = neighbours[0]
        p = random.random()
        if p <= (base_graph.degree(v))/(base_graph.degree(w)):
            subgraph_nodes.add(v)
            v = w
    return base_graph.subgraph(subgraph_nodes)

def mhda_dos(base_graph, n):
    subgraph_nodes = set()
    current = random_sample([v for v in base_graph.nodes()], 1)[0]
    pevious = None
    while len([n for n in subgraph_nodes]) < n:
        p = random.random()
        

def random_sample(collection: list[Any], n: int) -> list[Any]:
    if n >= len(collection):
        return collection
    s: set[Any] = set()
    while len(s) < n:
        random.shuffle(collection)
        s.add(collection.pop(0))
    return [x for x in s]

In [ ]:
from typing import List


def mhda_sample(base_graph: nx.DiGraph, n: int) -> nx.DiGraph:
    yt = None
    xt = random_sample([v for v in base_graph.nodes()], 1)[0]
    subgraph_nodes = set()
    while len(subgraph_nodes) < n:
        p = random.random()
        i = random_sample([n for n in base_graph.adj[xt[-1]]], 1)[0]
        if p <= min(1, base_graph.degree(xt)/base_graph.degree(i) ):
            if yt == i and base_graph.degree(xt) > 1:
                k = random_sample([n for n in base_graph.adj[xt] if n is not i], 1)[0]
                q = random.random()
                if q <= min(1, min(1, (base_graph.degree(xt)/base_graph.degree(k))**2)*max(1, (base_graph.degree(i)/base_graph.degree(xt))**2)): 
                    yt = xt 
                    xt = k
                else:
                    yt = xt
                    xt = i
            else:
                yt = xt
                xt = i
        subgraph_nodes.add(xt)
        print(len(subgraph_nodes))
    return base_graph.subgraph([n for n in subgraph_nodes])

def degree_weighted_random_choice_delayed_acceptance(
    start_node: Any, induced_subgraph: Set[Any], base: nx.DiGraph
):
    if start_node is None:
        # choosing a random node in the base based on degree probability
        listed = []
        for n in base.nodes():
            for _ in range(base.degree(n)):
                listed.append(n)
        random.shuffle(listed)
        if len(listed) == 0:
            return None
        return listed[0]
    # choose a random node weighted by degree from set of neighbours \ induced
    listed = []
    for n in base.adj[start_node]:
        if not (n in induced_subgraph):
            for _ in range(base.degree(n)):
                listed.append(n)
    random.shuffle(listed)
    if len(listed) == 0:
        return degree_weighted_random_choice_delayed_acceptance(None, induced_subgraph, base)
    return listed[0]

In [12]:
def sampled_comparison(input_data: nx.DiGraph):
  sampled_datasets = {
    "bfs": Sampling.bfs_sample(input_data, Sampling.random_sample([n for n in input_data.nodes()], 1), 1000),
   # "sns": Sampling.snowball_sample(input_data, Sampling.random_sample([n for n in input_data.nodes()], 1), 1000),
    "mhrw": mhrw(input_data, 1000),
    "mhda": mhda_sample(input_data, 1000)
  }
  return sampled_datasets

samples = sampled_comparison(enron_nx)
print(generate_row(samples['mhda']))

KeyboardInterrupt: 

In [ ]:
table = []
for method, data in [(method, generate_row(samples[method])) for method in ['bfs', 'sns', 'mhrw', 'mhda']]:
  table.append([method]+data)
raw_data = pd.DataFrame(table, columns=["Dataset", "Nodes", "Edges", "Mean Degree", "Clustering Coefficient", "Centralization"])
raw_data

ZeroDivisionError: division by zero